[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavinciDreams/SymbioGPT/blob/main/train_1m.ipynb)

# JuliaFluxGPT-1M: Scaling Law Data Point

Trains a 1.01M parameter LLaMA-style transformer on the curated philosophy corpus.
Same architecture family as JuliaFluxGPT-23M (GQA, RMSNorm, SwiGLU, RoPE) but
scaled down to provide a low-end data point for SLM scaling laws.

**Config**: d=128, 4 layers, 4Q/4KV (full MHA), head_dim=32, vocab=2000, ctx=256

**Chinchilla budget**: 20M tokens (20:1 ratio) → ~1,220 steps at batch=64

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup
!pip install -q wandb huggingface_hub
!git clone https://github.com/DavinciDreams/SymbioGPT.git /content/SymbioGPT 2>/dev/null || (cd /content/SymbioGPT && git pull)
%cd /content/SymbioGPT

In [ ]:
# 2. GPU check + imports
import os, sys, math, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, HfApi, create_repo

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 3. W&B + HF login
import os
import wandb
from huggingface_hub import login as hf_login

# Prefer Colab secrets, fall back to env vars, then interactive prompt
try:
    from google.colab import userdata
    os.environ.setdefault("WANDB_API_KEY", userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except (ImportError, Exception):
    pass  # Not in Colab or secrets not configured

wandb.login()
hf_login(token=os.environ.get("HF_TOKEN"), add_to_git_credential=False)

In [ ]:
# 4. Load pre-tokenized data
DATA_REPO = "LisaMegaWatts/SymbioGPT-10M"
os.makedirs("data", exist_ok=True)

print("Downloading pre-tokenized data (266M train, 72M val tokens)...")
hf_hub_download(repo_id=DATA_REPO, filename="data/train_curated.txt.tokens.pt", local_dir=".")
hf_hub_download(repo_id=DATA_REPO, filename="data/val.txt.tokens.pt", local_dir=".")

CTX = 256
print("Loading tokens...")
train_tokens = torch.load("data/train_curated.txt.tokens.pt", weights_only=True).tolist()
val_tokens = torch.load("data/val.txt.tokens.pt", weights_only=True).tolist()

def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f"Train: {len(train_inputs):,} seqs ({len(train_inputs)*CTX:,} tokens)")
print(f"Val: {len(val_inputs):,} seqs ({len(val_inputs)*CTX:,} tokens)")
del train_tokens, val_tokens

In [ ]:
# 5. Create 1M model
sys.path.insert(0, "/content/SymbioGPT")
from juliaflux_model import JuliaFluxConfig, JuliaFluxGPT

config = JuliaFluxConfig(
    d_model=128,
    n_layers=4,
    n_heads=4,
    n_kv_heads=4,     # full MHA (no GQA) at this scale
    head_dim=32,
    context_length=CTX,
    vocab_size=2000,
    weight_tying=True,
    rope_base=10000.0,
)

model = JuliaFluxGPT(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"JuliaFluxGPT-1M: {n_params:,} params ({n_params/1e6:.2f}M)")
print(f"Config: d={config.d_model}, L={config.n_layers}, H={config.n_heads}Q/{config.n_kv_heads}KV, hd={config.head_dim}")

# Verify forward pass
with torch.no_grad():
    test_x = torch.randint(0, config.vocab_size, (1, 32), device=device)
    test_logits = model(test_x)
    print(f"Forward pass: {test_x.shape} -> {test_logits.shape}")

In [ ]:
# 6. Training setup

# Chinchilla-optimal: 20M tokens for 1M params
# But we have 266M tokens — can train much longer to full convergence
BATCH_SIZE = 64
TOKENS_PER_STEP = BATCH_SIZE * CTX  # 16,384
CHINCHILLA_TOKENS = 20 * n_params   # ~20M
CHINCHILLA_STEPS = CHINCHILLA_TOKENS // TOKENS_PER_STEP  # ~1,220

# Train to 4x Chinchilla to ensure full convergence (data-rich regime)
TOTAL_STEPS = CHINCHILLA_STEPS * 4   # ~4,900 steps
WARMUP = 200
LR = 1e-3          # higher LR for smaller model
MIN_LR = 1e-4
EVAL_INTERVAL = 250
CHECKPOINT_INTERVAL = 1000

print(f"Chinchilla budget: {CHINCHILLA_TOKENS:,} tokens ({CHINCHILLA_STEPS} steps)")
print(f"Total training:    {TOTAL_STEPS * TOKENS_PER_STEP:,} tokens ({TOTAL_STEPS} steps)")
print(f"Data epochs:       {TOTAL_STEPS * TOKENS_PER_STEP / (len(train_inputs) * CTX):.1f}")

HF_REPO = "LisaMegaWatts/JuliaFluxGPT-1M"

# W&B init
run = wandb.init(
    project="symbiogenesis",
    name="juliafluxgpt-1m-scaling",
    config={
        "model": "JuliaFluxGPT-1M",
        "architecture": "LLaMA-style MHA",
        "d_model": config.d_model,
        "n_layers": config.n_layers,
        "n_heads": config.n_heads,
        "n_kv_heads": config.n_kv_heads,
        "head_dim": config.head_dim,
        "vocab_size": config.vocab_size,
        "context_length": CTX,
        "total_params": n_params,
        "batch_size": BATCH_SIZE,
        "total_steps": TOTAL_STEPS,
        "chinchilla_steps": CHINCHILLA_STEPS,
        "lr": LR,
        "min_lr": MIN_LR,
        "warmup": WARMUP,
        "weight_tying": True,
        "precision": "bf16",
        "purpose": "scaling_law_data_point",
    },
    tags=["scaling-laws", "1m", "juliafluxgpt", "philosophy"],
    reinit=True,
)
print(f"W&B: {run.url}")

In [ ]:
# 7. Evaluation function

def evaluate(model, val_inputs, val_labels, batch_size=128):
    """Compute val loss and perplexity."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    dev = next(model.parameters()).device
    with torch.no_grad():
        for i in range(0, len(val_inputs), batch_size):
            batch_in = val_inputs[i:i+batch_size].to(dev)
            batch_tgt = val_labels[i:i+batch_size].to(dev)
            logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction="sum"
            )
            total_loss += loss.item()
            total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl

# Pre-training baseline
init_loss, init_ppl = evaluate(model, val_inputs, val_labels)
print(f"Init: val_loss={init_loss:.4f} ppl={init_ppl:.1f}")

In [ ]:
# 8. Training loop

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95)
)

def lr_schedule(step):
    if step < WARMUP:
        return (step + 1) / max(WARMUP, 1)
    progress = (step - WARMUP) / max(TOTAL_STEPS - WARMUP, 1)
    return max(MIN_LR / LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

n_train = len(train_inputs)
model.train()
best_val_loss = float("inf")
best_step = 0
history = []
t_start = time.time()
step = 0

print(f"Training JuliaFluxGPT-1M for {TOTAL_STEPS} steps...")
print(f"Precision: {amp_dtype}, batch={BATCH_SIZE}, lr={LR}")

while step < TOTAL_STEPS:
    perm = torch.randperm(n_train)
    for i in range(0, n_train, BATCH_SIZE):
        if step >= TOTAL_STEPS:
            break

        idx = perm[i:i+BATCH_SIZE]
        batch_in = train_inputs[idx].to(device)
        batch_tgt = train_labels[idx].to(device)

        with torch.amp.autocast("cuda", enabled=True, dtype=amp_dtype):
            logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.reshape(B*T, V), batch_tgt.reshape(B*T))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        if step % 50 == 0:
            elapsed = time.time() - t_start
            tok_s = (step + 1) * TOKENS_PER_STEP / max(elapsed, 1)
            wandb.log({
                "train/loss": loss.item(),
                "train/lr": scheduler.get_last_lr()[0],
                "train/tokens_per_sec": tok_s,
            }, step=step)

        if step > 0 and step % EVAL_INTERVAL == 0:
            val_loss, val_ppl = evaluate(model, val_inputs, val_labels)
            history.append((step, val_loss, val_ppl))
            marker = " ** NEW BEST **" if val_loss < best_val_loss else ""
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_step = step
                # Save best checkpoint
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "config": {
                        "d_model": config.d_model, "n_layers": config.n_layers,
                        "n_heads": config.n_heads, "n_kv_heads": config.n_kv_heads,
                        "head_dim": config.head_dim, "context_length": CTX,
                        "vocab_size": config.vocab_size, "weight_tying": True,
                    },
                    "step": step, "val_loss": val_loss, "val_ppl": val_ppl,
                    "n_params": n_params,
                }, "juliaflux_1m_best.pt")
            wandb.log({"val/loss": val_loss, "val/perplexity": val_ppl}, step=step)
            elapsed = time.time() - t_start
            print(f"  [step {step:5d}] val_loss={val_loss:.4f} ppl={val_ppl:.1f} "
                  f"lr={scheduler.get_last_lr()[0]:.2e} ({elapsed:.0f}s){marker}")
            model.train()

        step += 1

# Final eval
final_loss, final_ppl = evaluate(model, val_inputs, val_labels)
history.append((step, final_loss, final_ppl))
if final_loss < best_val_loss:
    best_val_loss = final_loss
    best_step = step
    torch.save({
        "model_state_dict": model.state_dict(),
        "config": {
            "d_model": config.d_model, "n_layers": config.n_layers,
            "n_heads": config.n_heads, "n_kv_heads": config.n_kv_heads,
            "head_dim": config.head_dim, "context_length": CTX,
            "vocab_size": config.vocab_size, "weight_tying": True,
        },
        "step": step, "val_loss": final_loss, "val_ppl": final_ppl,
        "n_params": n_params,
    }, "juliaflux_1m_best.pt")

elapsed = time.time() - t_start
wandb.log({"val/final_loss": best_val_loss, "val/final_ppl": math.exp(min(best_val_loss, 20.0))})
print(f"\nTraining complete ({elapsed:.0f}s)")
print(f"Best: val_loss={best_val_loss:.4f} ppl={math.exp(min(best_val_loss, 20.0)):.1f} at step {best_step}")

In [ ]:
# 9. Results + scaling law context

print("\n" + "=" * 60)
print("SCALING LAW DATA POINTS — Curated Philosophy Corpus")
print("=" * 60)
print(f"\n{'Model':<20} {'Params':>10} {'Val Loss':>10} {'Val PPL':>10}")
print("-" * 52)
print(f"{'JuliaFluxGPT-1M':<20} {n_params:>10,} {best_val_loss:>10.4f} {math.exp(min(best_val_loss,20)):>10.1f}")
print(f"{'SymbioSLM':<20} {'4,070,000':>10} {'3.6200':>10} {'37.3':>10}")
print(f"{'MonarchSLM':<20} {'4,980,000':>10} {'3.6500':>10} {'38.4':>10}")
print(f"{'JuliaSLM':<20} {'5,040,000':>10} {'3.5400':>10} {'34.5':>10}")
print(f"{'SymbioGPT-10M':<20} {'11,053,400':>10} {'3.5600':>10} {'35.2':>10}")
print(f"\nTraining history:")
for step, loss, ppl in history:
    print(f"  step {step:5d}: loss={loss:.4f} ppl={ppl:.1f}")

wandb.finish()

In [ ]:
# 10. Upload to HuggingFace

hf_api = HfApi()
try:
    create_repo(HF_REPO, exist_ok=True)
    hf_api.upload_file(
        path_or_fileobj="juliaflux_1m_best.pt",
        path_in_repo="juliaflux_1m_best.pt",
        repo_id=HF_REPO,
        commit_message=f"Best checkpoint: val_loss={best_val_loss:.4f} ppl={math.exp(min(best_val_loss,20)):.1f} at step {best_step}"
    )
    # Also upload model definition
    hf_api.upload_file(
        path_or_fileobj="juliaflux_model.py",
        path_in_repo="juliaflux_model.py",
        repo_id=HF_REPO,
        commit_message="Model definition for loading checkpoint"
    )
    print(f"Uploaded to: https://huggingface.co/{HF_REPO}")
except Exception as e:
    print(f"HF upload failed: {e}")

print("Done!")